In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA
# Load dataset
data = load_breast_cancer()
X, y = data.data, data.target
# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
# Reduce to 2D using PCA for visualization
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X_pca, y, test_size=0.2, random_state=42)
# Train Gradient Boosting Classifier with subsample (stochastic boosting)
model = GradientBoostingClassifier(
   n_estimators=20,
   learning_rate=0.1,
   subsample=0.6,
   max_depth=3,
   random_state=42
)
model.fit(X_train, y_train)
# Predict and evaluate
y_pred = model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)
# Accuracy over boosting iterations
staged_preds = list(model.staged_predict(X_test))
accuracy_values = [accuracy_score(y_test, y_pred_i) for y_pred_i in staged_preds]
# Plot accuracy over iterations
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(accuracy_values) + 1), accuracy_values, label="Test Accuracy")
plt.axhline(y=test_accuracy, color='r', linestyle='--', label=f'Final Accuracy: {test_accuracy:.4f}')
plt.title("Test Accuracy over Boosting Iterations (Breast Cancer)")
plt.xlabel("Number of Trees")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()
# Plot decision boundary
x_min, x_max = X_pca[:, 0].min() - 1, X_pca[:, 0].max() + 1
y_min, y_max = X_pca[:, 1].min() - 1, X_pca[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                    np.linspace(y_min, y_max, 300))
Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, cmap=plt.cm.Paired, alpha=0.4)
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y, edgecolor='k', cmap=plt.cm.Paired)
plt.title("Decision Boundary (Stochastic Gradient Boosting - Breast Cancer)")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.grid(True)
plt.tight_layout()
plt.show()